> **Provenance.** Written during the author's research internship at the National Institute of Statistics (NIS), Cambodia, using Cambodia Agriculture Survey (CAS) 2024 holding-level survey data. **The survey data are confidential and are NOT included in this repository**, so this notebook cannot be run here as-is. All cell outputs have been removed. File paths were rewritten from private local paths to relative `data/...` paths during packaging; the analysis code is otherwise unchanged. See `data/README.md` for the data access statement.
>
> **This notebook.** Parcel-level rice **production** prediction, first version. **SUPERSEDED: this version has target leakage.** It builds `expected_production = area_harvested_ha x yield_total`, which equals the target, and keeps `yield_total` as a feature. Its recorded R-squared of 0.9998 is an artefact. It is kept only for the record; use notebook 06. See `reports/validity_review.md`. Original filename: `v5.ipynb`.


In [ ]:
# ============================================
# ULTIMATE Rice Production Prediction - FIXED VERSION
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingRegressor

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

# ==============================
# 1. Load Data
# ==============================
df = pd.read_csv("final_dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"Original target variable ranges:")
print(f"  production_total(kg): {df['production_total(kg)'].min():.2f} - {df['production_total(kg)'].max():.2f}")
print(f"  yield_total(kg/ha): {df['yield_total(kg/ha)'].min():.2f} - {df['yield_total(kg/ha)'].max():.2f}")

# ==============================
# 2. Enhanced Outlier Handling for PRODUCTION
# ==============================

# Drop identifiers only (keep production_total(kg) as target)
leakage_features = ["int_key", "int_id", "parcel_id", "crop_field"]
existing_leakage_features = [col for col in leakage_features if col in df.columns]
df = df.drop(columns=existing_leakage_features)
print(f"Dropped leakage features: {existing_leakage_features}")

def detect_outliers_iqr(data, feature):
    """Detect outliers using IQR method"""
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[feature] < lower_bound) | (data[feature] > upper_bound)]
    return lower_bound, upper_bound, len(outliers)

# Analyze outliers for key features
print("\n🔍 OUTLIER ANALYSIS:")
features_to_check = ['production_total(kg)', 'yield_total(kg/ha)', 'area_harvested_ha', 'stored_qty_total(kg)']

for feature in features_to_check:
    if feature in df.columns:
        lower, upper, n_outliers = detect_outliers_iqr(df, feature)
        print(f"{feature}:")
        print(f"  IQR range: {lower:.2f} - {upper:.2f}")
        print(f"  Actual range: {df[feature].min():.2f} - {df[feature].max():.2f}")
        print(f"  Number of outliers: {n_outliers} ({n_outliers/len(df)*100:.2f}%)")

# Production-specific outlier handling
original_count = len(df)

# Remove production outliers (based on your IQR analysis)
Q1_prod = df['production_total(kg)'].quantile(0.25)
Q3_prod = df['production_total(kg)'].quantile(0.75)
IQR_prod = Q3_prod - Q1_prod
lower_bound_prod = max(Q1_prod - 1.5 * IQR_prod, 0)  # Production can't be negative
upper_bound_prod = Q3_prod + 1.5 * IQR_prod

df_clean = df[(df['production_total(kg)'] >= lower_bound_prod) & 
              (df['production_total(kg)'] <= upper_bound_prod)].copy()

print(f"\n🎯 PRODUCTION OUTLIER HANDLING:")
print(f"Removed {original_count - len(df_clean)} production outliers")
print(f"New production range: {df_clean['production_total(kg)'].min():.2f} - {df_clean['production_total(kg)'].max():.2f}")

# Additional outlier removal for key features
outlier_features = ['area_harvested_ha', 'stored_qty_total(kg)', 'yield_total(kg/ha)']
for feature in outlier_features:
    if feature in df_clean.columns:
        Q1_f = df_clean[feature].quantile(0.25)
        Q3_f = df_clean[feature].quantile(0.75)
        IQR_f = Q3_f - Q1_f
        lower_f = max(Q1_f - 1.5 * IQR_f, 0)
        upper_f = Q3_f + 1.5 * IQR_f
        
        before = len(df_clean)
        df_clean = df_clean[(df_clean[feature] >= lower_f) & (df_clean[feature] <= upper_f)]
        print(f"Removed {before - len(df_clean)} outliers from {feature}")

print(f"Final dataset size: {len(df_clean)} samples ({len(df_clean)/original_count*100:.1f}% of original)")

# ==============================
# 3. PRODUCTION-SPECIFIC Feature Engineering
# ==============================

# Drop province if exists, keep zone
if "prov" in df_clean.columns:
    df_clean = df_clean.drop(columns=["prov"])

# 3.1 Production efficiency features
df_clean["harvest_ratio"] = df_clean["area_harvested_ha"] / (df_clean["area_planted_ha"] + 1e-6)
df_clean["loss_area"] = df_clean["area_planted_ha"] - df_clean["area_harvested_ha"]
df_clean["loss_ratio"] = df_clean["loss_area"] / (df_clean["area_planted_ha"] + 1e-6)

# 3.2 Production scaling features
df_clean["expected_production"] = df_clean["area_harvested_ha"] * df_clean["yield_total(kg/ha)"]
df_clean["production_efficiency"] = df_clean["production_total(kg)"] / (df_clean["expected_production"] + 1e-6)

# 3.3 Stress and problem indicators
stress_cols = ["reason_drought","reason_flood","reason_nat_event",
               "reason_pest","reason_theft","reason_sickness",
               "reason_no_labor","reason_other"]
existing_stress_cols = [col for col in stress_cols if col in df_clean.columns]
df_clean["stress_count"] = df_clean[existing_stress_cols].sum(axis=1)
df_clean["has_stress"] = (df_clean["stress_count"] > 0).astype(int)

# 3.4 Resource utilization features
df_clean["fert_irrig"] = df_clean["fert_applied"] * df_clean["irrigated"]
df_clean["pest_irrig"] = df_clean["pest_applied"] * df_clean["irrigated"]
df_clean["resource_intensity"] = df_clean["fert_applied"] + df_clean["pest_applied"] + df_clean["irrigated"]
df_clean["management_score"] = (df_clean["fert_applied"] * 0.4 + 
                               df_clean["pest_applied"] * 0.3 + 
                               df_clean["irrigated"] * 0.3)

# 3.5 Crop grouping
df_clean["crop_group"] = df_clean["crop_code"].replace({101:"NonAromatic",102:"Aromatic",103:"Sticky"})

# 3.6 Storage and post-harvest features
if "stored_qty_total(kg)" in df_clean.columns:
    df_clean["storage_ratio"] = df_clean["stored_qty_total(kg)"] / (df_clean["production_total(kg)"] + 1e-6)
    df_clean["marketed_ratio"] = 1 - df_clean["storage_ratio"]  # Rough estimate of what was sold/used

# 3.7 Area-based features
df_clean["area_productivity"] = df_clean["production_total(kg)"] / (df_clean["area_harvested_ha"] + 1e-6)
df_clean["large_scale_farm"] = (df_clean["area_harvested_ha"] > df_clean["area_harvested_ha"].median()).astype(int)

# 3.8 Log transformations for skewed production-related features
production_related_features = ['area_planted_ha', 'area_harvested_ha', 'stored_qty_total(kg)']
for feature in production_related_features:
    if feature in df_clean.columns:
        df_clean[f"log_{feature}"] = np.log1p(df_clean[feature])

print(f"Total features after production-specific engineering: {len(df_clean.columns)}")

# ==============================
# 4. Define Features & Target for PRODUCTION
# ==============================
target = "production_total(kg)"

y = df_clean[target]
X = df_clean.drop(columns=[target])

# ==============================
# 4.1 ROBUST Feature Selection
# ==============================

print("\n🔧 PERFORMING FEATURE SELECTION...")

# Get numeric columns that actually exist in X
numeric_cols = [col for col in X.select_dtypes(include=[np.number]).columns if col in X.columns]

# 1. Remove constant features
constant_features = []
for col in numeric_cols:
    if X[col].nunique() <= 1:
        constant_features.append(col)

if constant_features:
    print(f"Dropping constant features: {constant_features}")
    X = X.drop(columns=constant_features)
    # Update numeric_cols after dropping
    numeric_cols = [col for col in numeric_cols if col not in constant_features]

# 2. Remove highly correlated features (only if we have numeric columns)
if len(numeric_cols) > 1:
    try:
        correlation_matrix = X[numeric_cols].corr().abs()
        upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
        to_drop = []
        for column in upper_tri.columns:
            if any(upper_tri[column] > 0.90):
                to_drop.append(column)
        
        if to_drop:
            print(f"Dropping highly correlated features: {to_drop}")
            X = X.drop(columns=to_drop)
            numeric_cols = [col for col in numeric_cols if col not in to_drop]
    except Exception as e:
        print(f"Warning: Could not compute correlations: {e}")

# 3. Remove features with too many missing values (optional)
missing_threshold = 0.5  # Remove features with more than 50% missing
missing_ratios = X.isnull().sum() / len(X)
high_missing_features = missing_ratios[missing_ratios > missing_threshold].index.tolist()
if high_missing_features:
    print(f"Dropping features with >{missing_threshold*100}% missing values: {high_missing_features}")
    X = X.drop(columns=high_missing_features)

print(f"After feature selection: {X.shape[1]} features remaining")

# ==============================
# 5. Preprocessing for PRODUCTION
# ==============================
categorical_cols = ["zone","crop_group","fert_type_org","fert_type_inorg"]
categorical_cols = [col for col in categorical_cols if col in X.columns]

numerical_cols = [col for col in X.columns if col not in categorical_cols and X[col].dtype in ['int64', 'float64']]

print(f"\n📊 FINAL FEATURE SETUP:")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Numerical features: {len(numerical_cols)}")
print(f"Total features: {len(categorical_cols) + len(numerical_cols)}")

# Preprocessing pipelines
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numerical_cols),
        ("cat", cat_pipe, categorical_cols)
    ]
)

# ==============================
# 6. Train/Test Split
# ==============================
stratify_col = df_clean["zone"] if "zone" in df_clean.columns else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=stratify_col
)

print(f"\n🎯 DATA SPLIT:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training production range: {y_train.min():.0f} - {y_train.max():.0f} kg")
print(f"Test production range: {y_test.min():.0f} - {y_test.max():.0f} kg")

# ==============================
# 7. PRODUCTION-OPTIMIZED Models
# ==============================

# Models optimized for production prediction
models = {
    "LightGBM_Production": LGBMRegressor(
        n_estimators=500, 
        learning_rate=0.05, 
        max_depth=10,
        num_leaves=63, 
        subsample=0.8, 
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42, 
        n_jobs=-1,
        verbose=-1
    ),
    "XGBoost_Production": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest_Production": RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting_Production": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        max_features='sqrt',
        random_state=42
    )
}

# ==============================
# 8. Training & Evaluation for PRODUCTION
# ==============================
results = []
feature_importances = {}

print("\n" + "="*80)
print("TRAINING PRODUCTION PREDICTION MODELS")
print("="*80)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    try:
        pipe = Pipeline(steps=[("preprocessor", preprocessor),
                               ("model", model)])
        
        # Cross-validation
        cv_scores = cross_val_score(pipe, X_train, y_train, 
                                   cv=5, scoring='r2', n_jobs=1)
        
        # Full training
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Store feature importance
        if hasattr(pipe.named_steps['model'], 'feature_importances_'):
            preprocessor.fit(X_train)
            feature_names = numerical_cols.copy()
            if 'cat' in preprocessor.named_transformers_:
                cat_features = list(preprocessor.named_transformers_['cat']
                                  .named_steps['onehot'].get_feature_names_out(categorical_cols))
                feature_names.extend(cat_features)
            
            importances = pipe.named_steps['model'].feature_importances_
            feature_importances[name] = pd.DataFrame({
                'feature': feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)

        results.append([name, rmse, mae, r2, cv_scores.mean(), cv_scores.std()])
        print(f"{name} - R²: {r2:.4f}, CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"  RMSE: {rmse:.0f} kg, MAE: {mae:.0f} kg")
        
    except Exception as e:
        print(f"Error training {name}: {e}")
        continue

# ==============================
# 9. Final Production Model
# ==============================
if results:
    results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R²", "CV R² Mean", "CV R² Std"])
    print("\n" + "="*80)
    print("PRODUCTION MODEL PERFORMANCE COMPARISON")
    print("="*80)
    print(results_df.sort_values(by="R²", ascending=False).round(4))

    best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
    print(f"\n🏆 BEST PRODUCTION MODEL: {best_model_name}")

    # Train final model
    final_model = Pipeline(steps=[("preprocessor", preprocessor),
                                ("model", models[best_model_name])])
    final_model.fit(X_train, y_train)

    # Final predictions
    y_pred_final = final_model.predict(X_test)
    
    # Enhanced evaluation metrics
    from sklearn.metrics import mean_absolute_percentage_error
    
    final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
    final_mae = mean_absolute_error(y_test, y_pred_final)
    final_r2 = r2_score(y_test, y_pred_final)
    mape = mean_absolute_percentage_error(y_test, y_pred_final)
    
    # Calculate within 20% accuracy (more realistic for production)
    within_20_percent = np.mean(np.abs((y_test - y_pred_final) / y_test) <= 0.2) * 100

    print(f"\n🎯 FINAL PRODUCTION MODEL PERFORMANCE:")
    print(f"R²: {final_r2:.4f}")
    print(f"RMSE: {final_rmse:.0f} kg")
    print(f"MAE: {final_mae:.0f} kg")
    print(f"MAPE: {mape:.2%}")
    print(f"Within 20% accuracy: {within_20_percent:.1f}%")

    # ==============================
    # 10. Production Feature Analysis
    # ==============================
    print("\n" + "="*80)
    print("PRODUCTION FEATURE ANALYSIS")
    print("="*80)
    
    if best_model_name in feature_importances:
        important_features = feature_importances[best_model_name]
        
        print(f"\n📊 Top 15 Features for Production Prediction:")
        top_15 = important_features.head(15)
        for idx, row in top_15.iterrows():
            if row['feature'] in X.columns and X[row['feature']].dtype in ['int64', 'float64']:
                try:
                    corr = np.corrcoef(X[row['feature']], y)[0,1]
                    print(f"  {idx+1:2d}. {row['feature']}: {row['importance']:.1f} (corr: {corr:.3f})")
                except:
                    print(f"  {idx+1:2d}. {row['feature']}: {row['importance']:.1f}")
            else:
                print(f"  {idx+1:2d}. {row['feature']}: {row['importance']:.1f}")

    # ==============================
    # 11. Production Insights
    # ==============================
    print("\n" + "="*80)
    print("PRODUCTION PREDICTION INSIGHTS")
    print("="*80)
    
    residuals = y_test - y_pred_final
    residual_std = residuals.std()
    
    print(f"\n📈 Production Prediction Quality:")
    print(f"Average production: {y_test.mean():.0f} kg")
    print(f"Std of residuals: {residual_std:.0f} kg")
    print(f"95% of predictions within: ±{residual_std * 1.96:.0f} kg")
    
    # Production categories
    small_farms = y_test <= y_test.quantile(0.33)
    medium_farms = (y_test > y_test.quantile(0.33)) & (y_test <= y_test.quantile(0.66))
    large_farms = y_test > y_test.quantile(0.66)
    
    if small_farms.any():
        small_r2 = r2_score(y_test[small_farms], y_pred_final[small_farms])
        print(f"R² for small farms (<{y_test.quantile(0.33):.0f} kg): {small_r2:.3f}")
    
    if large_farms.any():
        large_r2 = r2_score(y_test[large_farms], y_pred_final[large_farms])
        print(f"R² for large farms (>{y_test.quantile(0.66):.0f} kg): {large_r2:.3f}")

print("\n" + "="*80)
print("🚀 PRODUCTION PREDICTION OPTIMIZATION COMPLETED!")
print("="*80)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

# ==============================
# DATA CORRELATION ANALYSIS
# ==============================

# Select numeric columns only
numeric_features = df.select_dtypes(include=['float64', 'int64']).columns
correlations = df[numeric_features].corr()['yield_total(kg/ha)'].sort_values(ascending=False)

print("Top 15 Positive Correlations with Yield:")
print(correlations[1:16])  # Skip first (correlation with itself)
print("\nTop 15 Negative Correlations with Yield:")
print(correlations[-15:])

# Create correlation matrix visualization
import matplotlib.pyplot as plt

# Select top correlated features
top_features = correlations.abs().nlargest(15).index
correlation_matrix = df[top_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, 
			annot=True, 
			cmap='coolwarm', 
			center=0, 
			fmt='.2f',
			square=True)
plt.title('Correlation Matrix of Top Features with Yield')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get numeric columns
numeric_features = df_clean.select_dtypes(include=['float64', 'int64']).columns

# Calculate correlations with target variable
correlations = df_clean[numeric_features].corr()['yield_total(kg/ha)'].sort_values(ascending=False)

# === 1. Bar plot of top correlations ===
plt.figure(figsize=(10, 6))
correlations[1:16].plot(kind='bar', color="skyblue")
plt.title('Top 15 Positive Correlations with Yield', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Correlation Coefficient')
plt.show()

# === 2. Bar plot of bottom correlations ===
plt.figure(figsize=(10, 6))
correlations[-10:].plot(kind='bar', color="salmon")
plt.title('Top 10 Negative Correlations with Yield', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Correlation Coefficient')
plt.show()

# === 3. Heatmap of top correlated features ===
plt.figure(figsize=(12, 8))
top_features = correlations.abs().nlargest(15).index
correlation_matrix = df_clean[top_features].corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    square=True
)
plt.title('Correlation Matrix of Top 15 Features', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

# Print summary
print("\nTop 10 Positive Correlations with Yield:")
print(correlations[1:11])

print("\nTop 10 Negative Correlations with Yield:")
print(correlations[-10:])


In [ ]:
# Create a DataFrame with actual and predicted values
predictions_df = pd.DataFrame({
	'actual_production': y_test,
	'predicted_production': y_pred_final,
	'absolute_error': abs(y_test - y_pred_final),
	'percentage_error': abs((y_test - y_pred_final) / y_test) * 100
})

# Add some additional analysis
predictions_df['within_20_percent'] = predictions_df['percentage_error'] <= 20

# Save to CSV
predictions_df.to_csv('production_predictions.csv', index=True)

print("Predictions saved to 'production_predictions.csv'")
print(f"\nSummary:")
print(f"Total predictions: {len(predictions_df)}")
print(f"Mean absolute error: {predictions_df['absolute_error'].mean():.2f} kg")
print(f"Mean percentage error: {predictions_df['percentage_error'].mean():.2f}%")
print(f"Predictions within 20%: {predictions_df['within_20_percent'].mean()*100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Define different error thresholds to analyze
thresholds = [5, 10, 15, 20, 25, 30]

print("Production Prediction Accuracy Analysis")
print("=====================================")

for threshold in thresholds:
	# Calculate percentage error
	within_threshold = np.abs((predictions_df['actual_production'] - predictions_df['predicted_production']) / predictions_df['actual_production'] * 100) <= threshold
	
	accuracy = within_threshold.mean() * 100
	
	print(f"\nWithin {threshold}% error:")
	print(f"Accuracy: {accuracy:.2f}%")
	print(f"Count: {within_threshold.sum()} out of {len(predictions_df)}")

# Analyze over/under predictions
over_predicted = predictions_df['predicted_production'] > predictions_df['actual_production']
under_predicted = predictions_df['predicted_production'] < predictions_df['actual_production']

print("\nPrediction Bias Analysis")
print("=====================")
print(f"Over-predicted: {over_predicted.sum()} ({over_predicted.mean()*100:.2f}%)")
print(f"Under-predicted: {under_predicted.sum()} ({under_predicted.mean()*100:.2f}%)")

# Calculate mean error for over/under predictions
mean_over_error = predictions_df[over_predicted]['percentage_error'].mean()
mean_under_error = predictions_df[under_predicted]['percentage_error'].mean()

print(f"\nMean error when over-predicting: {mean_over_error:.2f}%")
print(f"Mean error when under-predicting: {mean_under_error:.2f}%")

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

# Create figure with larger size
plt.figure(figsize=(12, 6))

# Create box plot
data_to_plot = [predictions_df['actual_production'], predictions_df['predicted_production']]
labels = ['Actual Production', 'Predicted Production']

# Create box plot using seaborn for better styling
sns.boxplot(data=data_to_plot)

# Customize the plot
plt.xticks([0, 1], labels)
plt.ylabel('Production (kg)')
plt.title('Distribution of Actual vs Predicted Rice Production')

# Add grid for better readability
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

# Tight layout to prevent label cutoff
plt.tight_layout()

# Show the plot
plt.show()